In [1]:
! /usr/local/bin/python3.12 -m pip install tensorflow keras


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import numpy
import pandas

from matplotlib import pyplot
import seaborn

# Redes neuronales
import keras

In [3]:
titanic3 = pandas.read_csv("../conjuntos/titanic3.csv")

titanic3

,Sex,Age,Pclass2,Family,Family2,Fare2,Cabin2,Embarked2,Survived2
0,male,22.0,3ra,1,couple,2.110213,no,S,no
1,female,38.0,1ra,1,couple,4.280593,yes,C,yes
2,female,26.0,3ra,0,single,2.188856,no,S,yes
3,female,35.0,1ra,1,couple,3.990834,yes,S,yes
4,male,35.0,3ra,0,single,2.202765,no,S,no
...,...,...,...,...,...,...,...,...,...
886,male,27.0,2da,0,single,2.639057,no,S,no
887,female,19.0,1ra,0,single,3.433987,yes,S,yes
888,female,NaN,3ra,3,family,3.196630,no,S,no
889,male,26.0,1ra,0,single,3.433987,yes,C,yes


In [4]:
pandas.__version__

'2.2.2'

In [5]:
pandas.get_dummies(titanic3[["Pclass2"]])[["Pclass2_1ra", "Pclass2_2da"]].astype(int)

,Pclass2_1ra,Pclass2_2da
0,0,0
1,1,0
2,0,0
3,1,0
4,0,0
...,...,...
886,0,1
887,1,0
888,0,0
889,1,0


In [20]:
x1 = (titanic3["Sex"] == "female").astype(int)
x1.name = "Sex_female"
x2 = titanic3["Age"]
x3 = (titanic3["Pclass2"] == "1ra").astype(int)
x3.name = "Pclass2_1ra"
x4 = (titanic3["Pclass2"] == "2da").astype(int)
x4.name = "Pclass2_2da"
x5 = titanic3["Family"]
x6 = (titanic3["Family2"] == "couple").astype(int)
x6.name = "Family2_couple"
x7 = (titanic3["Family2"] == "family").astype(int)
x7.name = "Family2_family"
x8 = titanic3["Fare2"]
x9 = (titanic3["Cabin2"] == "yes").astype(int)
x10 = (titanic3["Embarked2"] == "C").astype(int)
x10.name = "Embarked2_C"
x11 = (titanic3["Embarked2"] == "Q").astype(int)
x11.name = "Embarked2_Q"
x12 = (titanic3["Embarked2"] == "X").astype(int)
x12.name = "Embarked2_X"

y1 = (titanic3["Survived2"] == "yes").astype(int)

indices_edades_no_nulas = x2.dropna().index # Índices de las edades no conocidas
indices_edades_nulas = x2[x2.isna()].index # Índices de las edades conocidas
Cy = x2.dropna() # Edades conocidas
Cx = pandas.DataFrame([x1, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, y1]).T.loc[indices_edades_no_nulas] # Clúster de edades conocidas
Cp = pandas.DataFrame([x1, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, y1]).T.loc[indices_edades_nulas] # Clúster de predicción (para las edades no conocidas)

from sklearn.cluster import KMeans # MeanShift, AffinityPropagation

clu = KMeans(n_clusters=20) # Reconocer 20 tipos de clúster
clu.fit(Cx, Cy) # Entrenar el clúster
clusters = clu.predict(Cx) # Predecir el clúster para las edades conocidas
print(clusters)
age_cluster = pandas.DataFrame({
    "Age": Cy.values, # Edades conocidas
    "Clu": clusters, # Clúster de las edades conocidas
}).groupby(["Clu"]).median() # Agrupadas y se extrae la mediana (para las edades conocidas)

print(age_cluster)

X = pandas.DataFrame([x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12]).T # Las predictivas (sin imputar)
Y = pandas.DataFrame([y1]).T # Las respuestas

print(age_cluster.loc[clu.predict(Cp)]["Age"].values)

X.loc[x2.isna(), "Age"] = age_cluster.loc[clu.predict(Cp)]["Age"].values # Imputación de edades faltas por su edad mediana del clúster

X

[10 17 11 17  7  2  8  4 19  4  2  7  3  6  9  8 10  0  9 11  2 12  3  1
  2  0  5  5  7  4 16 10 19 12 11 10  8  7 17 19  5  9 18 12  3 18 13  5
  8  9  7  3  4  0  3  0 10  2  7 11  7 14  2  9 12  8  1  7  7  7  5  4
  7  2 17 19 19  6  5  7  4  7 11  7  2 16  7 10 18  7  7 19  5  3  4 19
  9  5 16 14  7 18  7 10 19  0  0 15  5  7  2 11 16  7  0  4 14 12  4  0
  0 17  7  4  5 11  7 10  9  7  7  8  4  8  2  2  8  4  7  2  4  2  0  7
  3 12  4  2  4  7  9  0 16  4  2 13 10  0  7  7 18 14 10 10 14 11  2  7
  9  7  0 17 11 19 13  0 14  0  7  5  7 14  7  0 17  7  0  3  0 19  4  0
  0  0  7 18 15  6  4 15 19  4  2 10  4  4 13 13 19  3 15  7  0  8 10 17
 13 14 19  5 17  6  8  4  7  7  7 14 18 14  7 14 11 13 17  0  6  7 18  1
 17  7 17 19 13 13  1  4  7  4 11 19  0 15 15  7  7  9  4 13  7  9  4 17
  2  5  4  5 13 14  2  4  1  0  0  0  9  9  4  7  7  4 10  7 17  0  8 19
 16  7  7 17 13 17 10  7 13 12 11 15 18  7 13  4  7 17  0  3  9  9  1 14
  4 17  4  7  6  0  0  9 14  7 10 10  6 19  7  4  7

,Sex_female,Age,Pclass2_1ra,Pclass2_2da,Family,Family2_couple,Family2_family,Fare2,Cabin2,Embarked2_C,Embarked2_Q,Embarked2_X
0,0.0,22.0,0.0,0.0,1.0,1.0,0.0,2.110213,0.0,0.0,0.0,0.0
1,1.0,38.0,1.0,0.0,1.0,1.0,0.0,4.280593,1.0,1.0,0.0,0.0
2,1.0,26.0,0.0,0.0,0.0,0.0,0.0,2.188856,0.0,0.0,0.0,0.0
3,1.0,35.0,1.0,0.0,1.0,1.0,0.0,3.990834,1.0,0.0,0.0,0.0
4,0.0,35.0,0.0,0.0,0.0,0.0,0.0,2.202765,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,27.0,0.0,1.0,0.0,0.0,0.0,2.639057,0.0,0.0,0.0,0.0
887,1.0,19.0,1.0,0.0,0.0,0.0,0.0,3.433987,1.0,0.0,0.0,0.0
888,1.0,24.0,0.0,0.0,3.0,0.0,1.0,3.196630,0.0,0.0,0.0,0.0
889,0.0,26.0,1.0,0.0,0.0,0.0,0.0,3.433987,1.0,1.0,0.0,0.0


In [10]:
numpy.exp(2.11 - 0 - 1)

np.float64(3.0343583944356753)

In [9]:
Y

,Survived2
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


## Conjuntos de Entrenamiento y Validación del Modelo de Aprendizaje

In [22]:
X.shape, Y.shape

((891, 12), (891, 1))

In [24]:
from sklearn import model_selection

Xa, Xb, Ya, Yb = model_selection.train_test_split(X, Y, test_size=100, random_state=123)

Xa.shape, Xb.shape, Ya.shape, Yb.shape

((791, 12), (100, 12), (791, 1), (100, 1))

In [25]:
X.to_csv("../conjuntos/titanic_X.csv")
Xa.to_csv("../conjuntos/titanic_X_train.csv")
Xb.to_csv("../conjuntos/titanic_X_test.csv")
Y.to_csv("../conjuntos/titanic_Y.csv")
Ya.to_csv("../conjuntos/titanic_Y_train.csv")
Yb.to_csv("../conjuntos/titanic_Y_test.csv")